# View Ray Tracer PGM Output

Load and display PGM (P5 binary) images produced by the ray tracer.
Adjust the `data_dir` and `glob_pattern` below to match your run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

In [ ]:
def read_pgm_p5(filepath):
    """Read a P5 (binary) PGM file. Returns (width, height, maxval, pixel_array)."""
    with open(filepath, "rb") as f:
        header = f.readline().strip()
        if header != b"P5":
            raise ValueError(f"Not a P5 PGM file: {header}")
        # skip comment lines
        line = f.readline()
        while line.startswith(b"#"):
            line = f.readline()
        width, height = map(int, line.split())
        maxval = int(f.readline().strip())
        pixels = np.frombuffer(f.read(), dtype=np.uint8 if maxval < 256 else np.uint16)
        return width, height, maxval, pixels.reshape((height, width))

def find_pgm_files(data_dir, pattern="rt_image_*.pgm"):
    """Find PGM files matching pattern, sorted by step number."""
    data_dir = Path(data_dir)
    files = sorted(data_dir.glob(pattern), key=lambda p: int(re.search(r"(\d+)", p.stem).group()))
    return files

def show_image(ax, filepath, title=None, cmap="inferno"):
    """Display a single PGM on an axes."""
    w, h, maxval, img = read_pgm_p5(str(filepath))
    im = ax.imshow(img.T, origin="lower", cmap=cmap, aspect="equal")
    ax.set_title(title or filepath.name)
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    plt.colorbar(im, ax=ax, label="brightness")
    return im

### Find available PGM files

In [ ]:
# --- Adjust these ---
data_dir = "Data"             # or "." if PGM files are in cwd
pattern = "rt_image_*.pgm"
# -------------------

files = find_pgm_files(data_dir, pattern)
print(f"Found {len(files)} PGM file(s):")
for f in files:
    w, h, maxval, _ = read_pgm_p5(str(f))
    print(f"  {f.name}  ({w}x{h}, maxval={maxval})")

### View a single frame

Set `index` to choose which frame (0 = first, -1 = last).

In [ ]:
index = 0  # change to view different frames (0 = first, -1 = last)
filepath = files[index]
w, h, maxval, img = read_pgm_p5(str(filepath))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(img.T, origin="lower", cmap="inferno", aspect="equal")
ax.set_title(f"{filepath.name}  ({w}x{h})")
ax.set_xlabel("x pixel"); ax.set_ylabel("y pixel")
plt.colorbar(im, ax=ax, label="line-of-sight brightness")
plt.tight_layout()
plt.show()

### View all frames side-by-side (up to first 16)

In [ ]:
n = min(len(files), 16)
cols = min(n, 4)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.5*rows))
axes = np.atleast_1d(axes).flatten()

# find global max for consistent scaling
global_max = 0
for f in files[:n]:
    _, _, _, img = read_pgm_p5(str(f))
    global_max = max(global_max, img.max())
global_max = max(global_max, 1)

for i, f in enumerate(files[:n]):
    _, _, _, img = read_pgm_p5(str(f))
    im = axes[i].imshow(img.T, origin="lower", cmap="inferno", aspect="equal",
                        vmin=0, vmax=global_max)
    step = re.search(r"(\d+)", f.stem).group()
    axes[i].set_title(f"step {step}")
    axes[i].set_xlabel("x"); axes[i].set_ylabel("y")

# hide unused subplots
for j in range(n, len(axes)):
    axes[j].set_visible(False)

if n > 1:
    fig.colorbar(im, ax=list(axes[:n]), label="brightness", shrink=0.8)
plt.tight_layout()
plt.show()